<a href="https://colab.research.google.com/github/izzajavaid-svg/Telco_churn/blob/main/Telco_Churn_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML Fundamentals & Regression Models
## IBM Telco Customer Churn

In [2]:
# ============================================
# STEP 1 — IMPORT LIBRARIES
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

RANDOM_SEED = 42

print("Libraries imported successfully.")
print("Random seed:", RANDOM_SEED)

Libraries imported successfully.
Random seed: 42


In [3]:
# ============================================
# STEP 2 — LOAD DATASET
# ============================================

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# ============================================
# STEP 3 — INITIAL DATA INSPECTION
# ============================================

print("Dataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nTarget distribution:")
display(df["Churn"].value_counts())

print("\nTarget proportions:")
display(df["Churn"].value_counts(normalize=True))

Dataset shape:
(7043, 21)

Column names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data types:


,0
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object



Missing values:


,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0



Duplicate rows:
0

Target distribution:


,count
Churn,
No,5174
Yes,1869



Target proportions:


,proportion
Churn,
No,0.73463
Yes,0.26537


In [5]:
# ============================================
# STEP 4 — CHECK NUMERICAL FEATURES
# ============================================

print("TotalCharges dtype before conversion:")
print(df["TotalCharges"].dtype)

print("\nSample TotalCharges values:")
display(df["TotalCharges"].head(10))

TotalCharges dtype before conversion:
object

Sample TotalCharges values:


,TotalCharges
0,29.85
1,1889.5
2,108.15
3,1840.75
4,151.65
5,820.5
6,1949.4
7,301.9
8,3046.05
9,3487.95


In [6]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

print("TotalCharges dtype after conversion:")
print(df["TotalCharges"].dtype)

print("\nMissing TotalCharges after conversion:")
print(df["TotalCharges"].isnull().sum())

TotalCharges dtype after conversion:
float64

Missing TotalCharges after conversion:
11


In [7]:
# ============================================
# B3 — PREPARE BASELINE FEATURES
# ============================================

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("TotalCharges dtype:", df["TotalCharges"].dtype)
print("Missing TotalCharges:", df["TotalCharges"].isna().sum())

TotalCharges dtype: float64
Missing TotalCharges: 11


In [8]:
baseline_features = ["tenure", "MonthlyCharges", "TotalCharges"]

X = df[baseline_features].copy()

y = (df["Churn"] == "Yes").astype(int)

print("Features:")
display(X.head())

print("\nFeature data types:")
print(X.dtypes)

print("\nMissing values:")
print(X.isna().sum())

print("\nTarget:")
print(y.head())

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Features:


,tenure,MonthlyCharges,TotalCharges
0,1,29.85,29.85
1,34,56.95,1889.50
2,2,53.85,108.15
3,45,42.30,1840.75
4,2,70.70,151.65



Feature data types:
tenure              int64
MonthlyCharges    float64
TotalCharges      float64
dtype: object

Missing values:
tenure             0
MonthlyCharges     0
TotalCharges      11
dtype: int64

Target:
0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64

X shape: (7043, 3)
y shape: (7043,)


In [9]:
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

print("\nOverall churn proportion:", y.mean())
print("Training churn proportion:", y_train.mean())
print("Test churn proportion:", y_test.mean())

Training set size: (5634, 3)
Test set size: (1409, 3)

Overall churn proportion: 0.2653698707936959
Training churn proportion: 0.2653532126375577
Test churn proportion: 0.2654364797728886


In [10]:
# ============================================
# B3 — BASELINE LOGISTIC REGRESSION
# ============================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(max_iter=1000))
])

# Train only on the training set
baseline_model.fit(X_train, y_train)

print("Baseline Logistic Regression trained successfully!")

Baseline Logistic Regression trained successfully!


In [11]:
# ============================================
# B3 — BASELINE EVALUATION
# ============================================

from sklearn.metrics import classification_report, confusion_matrix

# Predictions
y_pred = baseline_model.predict(X_test)

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["No Churn", "Churn"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

    No Churn       0.81      0.90      0.85      1035
       Churn       0.60      0.42      0.49       374

    accuracy                           0.77      1409
   macro avg       0.71      0.66      0.67      1409
weighted avg       0.76      0.77      0.76      1409

Confusion Matrix:
[[931 104]
 [217 157]]


In [12]:
# ============================================
# B4 — 5-FOLD STRATIFIED CROSS-VALIDATION
# ============================================

from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED
)

cv_scores = cross_val_score(
    baseline_model,
    X,
    y,
    cv=cv,
    scoring="f1"
)

print("F1 scores for each fold:")
for i, score in enumerate(cv_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print("\nMean F1:", cv_scores.mean())
print("Standard deviation:", cv_scores.std())
print(f"\nMean F1 ± Std: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

F1 scores for each fold:
Fold 1: 0.5483
Fold 2: 0.5237
Fold 3: 0.5506
Fold 4: 0.4913
Fold 5: 0.4846

Mean F1: 0.5197193829095454
Standard deviation: 0.027686996133717733

Mean F1 ± Std: 0.5197 ± 0.0277


In [13]:
# ============================================
# B1 — FEATURE ENGINEERING
# ============================================

# 1. Tenure buckets
df["tenure_bucket"] = pd.cut(
    df["tenure"],
    bins=[-1, 12, 24, 48, 72],
    labels=False
)

# 2. Monthly charge relative to tenure
df["charge_ratio"] = df["MonthlyCharges"] / (df["tenure"] + 1)

print("Engineered features created successfully!")

display(
    df[
        [
            "tenure",
            "MonthlyCharges",
            "TotalCharges",
            "Contract",
            "tenure_bucket",
            "charge_ratio"
        ]
    ].head(10)
)

Engineered features created successfully!


,tenure,MonthlyCharges,TotalCharges,Contract,tenure_bucket,charge_ratio
0,1,29.85,29.85,Month-to-month,0,14.925000
1,34,56.95,1889.50,One year,2,1.627143
2,2,53.85,108.15,Month-to-month,0,17.950000
3,45,42.30,1840.75,One year,2,0.919565
4,2,70.70,151.65,Month-to-month,0,23.566667
5,8,99.65,820.50,Month-to-month,0,11.072222
6,22,89.10,1949.40,Month-to-month,1,3.873913
7,10,29.75,301.90,Month-to-month,0,2.704545
8,28,104.80,3046.05,Month-to-month,2,3.613793
9,62,56.15,3487.95,One year,3,0.891270


In [14]:
# ============================================
# DEFINE ENGINEERED FEATURES
# ============================================

numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "tenure_bucket",
    "charge_ratio"
]

categorical_features = [
    "Contract"
]

X_engineered = df[numeric_features + categorical_features].copy()

print("Engineered feature set:")
display(X_engineered.head())

print("\nShape:", X_engineered.shape)

print("\nMissing values:")
print(X_engineered.isna().sum())

Engineered feature set:


,tenure,MonthlyCharges,TotalCharges,tenure_bucket,charge_ratio,Contract
0,1,29.85,29.85,0,14.925000,Month-to-month
1,34,56.95,1889.50,2,1.627143,One year
2,2,53.85,108.15,0,17.950000,Month-to-month
3,45,42.30,1840.75,2,0.919565,One year
4,2,70.70,151.65,0,23.566667,Month-to-month



Shape: (7043, 6)

Missing values:
tenure             0
MonthlyCharges     0
TotalCharges      11
tenure_bucket      0
charge_ratio       0
Contract           0
dtype: int64


In [15]:
# ============================================
# TRAIN / TEST SPLIT
# ============================================

X_train_eng, X_test_eng, y_train_eng, y_test_eng = train_test_split(
    X_engineered,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED
)

print("Training set:", X_train_eng.shape)
print("Test set:", X_test_eng.shape)

print("\nTraining churn proportion:", y_train_eng.mean())
print("Test churn proportion:", y_test_eng.mean())

Training set: (5634, 6)
Test set: (1409, 6)

Training churn proportion: 0.2653532126375577
Test churn proportion: 0.2654364797728886


In [16]:
# ============================================
# PREPROCESSING + LOGISTIC REGRESSION
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ]
)

engineered_model = Pipeline([
    ("preprocessor", preprocessor),
    ("logistic_regression", LogisticRegression(max_iter=1000))
])

engineered_model.fit(
    X_train_eng,
    y_train_eng
)

print("Engineered Logistic Regression trained successfully!")

Engineered Logistic Regression trained successfully!


In [17]:
# ============================================
# B1 — EVALUATE ENGINEERED MODEL
# ============================================

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

# Predictions
y_pred_eng = engineered_model.predict(X_test_eng)

print("Engineered Model Classification Report:")
print(classification_report(
    y_test_eng,
    y_pred_eng,
    target_names=["No Churn", "Churn"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test_eng, y_pred_eng))

# Churn-specific metrics
eng_precision = precision_score(y_test_eng, y_pred_eng)
eng_recall = recall_score(y_test_eng, y_pred_eng)
eng_f1 = f1_score(y_test_eng, y_pred_eng)

print("\nChurn metrics:")
print(f"Precision: {eng_precision:.4f}")
print(f"Recall:    {eng_recall:.4f}")
print(f"F1 Score:  {eng_f1:.4f}")

Engineered Model Classification Report:
              precision    recall  f1-score   support

    No Churn       0.82      0.91      0.86      1035
       Churn       0.65      0.45      0.53       374

    accuracy                           0.79      1409
   macro avg       0.73      0.68      0.70      1409
weighted avg       0.78      0.79      0.78      1409

Confusion Matrix:
[[942  93]
 [204 170]]

Churn metrics:
Precision: 0.6464
Recall:    0.4545
F1 Score:  0.5338


Feature engineering improved the Logistic Regression baseline. Churn F1 increased from 0.49 to 0.5338, while precision increased from 0.60 to 0.6464 and recall increased from 0.42 to 0.4545. Accuracy also increased from 0.77 to 0.79. The engineered model reduced false negatives from 217 to 204 and increased true positives from 157 to 170.

In [18]:
# ============================================
# B2 — ERROR ANALYSIS
# ============================================

# Find misclassified customers
errors = X_test_eng.copy()

errors["Actual_Churn"] = y_test_eng
errors["Predicted_Churn"] = y_pred_eng

misclassified = errors[
    errors["Actual_Churn"] != errors["Predicted_Churn"]
]

print("Total misclassified customers:", len(misclassified))

# Randomly inspect 10 misclassified examples
sample_errors = misclassified.sample(
    10,
    random_state=RANDOM_SEED
)

display(sample_errors)

Total misclassified customers: 297


,tenure,MonthlyCharges,TotalCharges,tenure_bucket,charge_ratio,Contract,Actual_Churn,Predicted_Churn
3803,61,94.10,5638.30,3,1.517742,Two year,1,0
3154,4,72.75,317.75,0,14.550000,Month-to-month,0,1
4653,30,51.20,1561.50,2,1.651613,Month-to-month,1,0
113,37,76.50,2868.15,2,2.013158,Month-to-month,1,0
761,22,89.25,1907.85,1,3.880435,Month-to-month,1,0
2005,49,100.45,4941.80,3,2.009000,One year,1,0
4912,36,90.85,3186.70,2,2.455405,Month-to-month,1,0
1446,1,50.05,50.05,0,25.025000,Month-to-month,0,1
3510,8,75.25,576.70,0,8.361111,Month-to-month,0,1
6157,3,19.85,64.55,0,4.962500,Month-to-month,1,0


In [19]:
# Add predicted probabilities
sample_errors = sample_errors.copy()

sample_errors["Churn_Probability"] = engineered_model.predict_proba(
    X_test_eng.loc[sample_errors.index]
)[:, 1]

display(
    sample_errors[
        [
            "tenure",
            "MonthlyCharges",
            "TotalCharges",
            "tenure_bucket",
            "charge_ratio",
            "Contract",
            "Actual_Churn",
            "Predicted_Churn",
            "Churn_Probability"
        ]
    ].sort_values("Churn_Probability")
)

,tenure,MonthlyCharges,TotalCharges,tenure_bucket,charge_ratio,Contract,Actual_Churn,Predicted_Churn,Churn_Probability
3803,61,94.10,5638.30,3,1.517742,Two year,1,0,0.043923
2005,49,100.45,4941.80,3,2.009000,One year,1,0,0.177307
4653,30,51.20,1561.50,2,1.651613,Month-to-month,1,0,0.209355
6157,3,19.85,64.55,0,4.962500,Month-to-month,1,0,0.216189
113,37,76.50,2868.15,2,2.013158,Month-to-month,1,0,0.313930
4912,36,90.85,3186.70,2,2.455405,Month-to-month,1,0,0.403670
761,22,89.25,1907.85,1,3.880435,Month-to-month,1,0,0.475756
3510,8,75.25,576.70,0,8.361111,Month-to-month,0,1,0.511671
1446,1,50.05,50.05,0,25.025000,Month-to-month,0,1,0.578906
3154,4,72.75,317.75,0,14.550000,Month-to-month,0,1,0.581007


Failure mode: The model produces false negatives for some customers who have long tenure or longer contracts. For example, one actual churner with 61 months of tenure and a two-year contract received only a 4.4% predicted churn probability. This suggests that the current feature set does not capture all factors associated with churn, particularly factors that may cause otherwise stable customers to leave.

In [20]:
# ============================================
# PART C — LEARNING CURVE
# ============================================

from sklearn.model_selection import learning_curve
import numpy as np
import matplotlib.pyplot as plt

train_sizes, train_scores, validation_scores = learning_curve(
    estimator=engineered_model,
    X=X_engineered,
    y=y,
    cv=cv,
    scoring="f1",
    train_sizes=np.linspace(0.1, 1.0, 5),
    n_jobs=-1
)

# Calculate mean and standard deviation
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)

validation_mean = validation_scores.mean(axis=1)
validation_std = validation_scores.std(axis=1)

print("Training sizes:")
print(train_sizes)

print("\nTraining F1:")
print(train_mean)

print("\nValidation F1:")
print(validation_mean)

Training sizes:
[ 563 1831 3098 4366 5634]

Training F1:
[0.49879565 0.55323004 0.54600822 0.53094811 0.53381561]

Validation F1:
[0.47523878 0.5299124  0.54144749 0.53418359 0.53306124]


The learning curves show relatively small differences between training and validation F1, indicating no strong evidence of overfitting. Both scores converge around 0.53 as the training set increases, suggesting reasonable generalization but also that the current model and feature set may have reached a performance plateau.